In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CENSUS_DIR = DATA_DIR / "census" / "processed"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

tracts_acs_path = CENSUS_DIR / "tracts_acs_2019_2023.gpkg"
village_clean_path = BOUNDARIES_DIR / "processed" / "village_clean.gpkg"

tracts_acs_path, village_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/processed/tracts_acs_2019_2023.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/village_clean.gpkg'))

In [2]:
tracts = gpd.read_file(tracts_acs_path)
village = gpd.read_file(village_clean_path)

print("Tracts CRS:", tracts.crs)
print("Village CRS:", village.crs)

Tracts CRS: EPSG:2868
Village CRS: EPSG:2868


In [3]:
# anxietying CRS differences

PROJECT_CRS = "EPSG:2868"

if village.crs != PROJECT_CRS:
    village = village.to_crs(PROJECT_CRS)
if tracts.crs != PROJECT_CRS:
    tracts = tracts.to_crs(PROJECT_CRS)

tracts.crs, village.crs

(<Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich,
 <Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - 

In [4]:
tracts.columns

Index(['GISJOIN', 'ASNQE001', 'ASQPE001', 'ASVBE001', 'ASS8E001', 'ASS8E002',
       'ASS8E003', 'ASS9E001', 'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003',
       'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002',
       'geometry'],
      dtype='object')

In [5]:
village.columns

Index(['OBJECTID', 'NAME', 'geometry'], dtype='object')

In [6]:
TRACT_GEOID_FIELD = "GISJOIN"
TOTAL_POPULATION = "ASNQE001"         # Total population (Sex by Age: total)
MEDIAN_HOUSEHOLD_INCOME = "ASQPE001"     # Median household income (2023 inflation-adjusted $)
MEDIAN_RENT = "ASVBE001"       # Median gross rent (dollars)
HOUSING_TOTAL = "ASS8E001"  # Total housing units
OCCUPIED_HOUSEHOLDS = "ASS8E002"     # Total occupied housing units
VACANT_UNITS = "ASS8E003"     # Vacant housing units
OWNER_OCCUPIED = "ASS9E002"     # Owner-occupied
RENTER_OCCUPIED = "ASS9E003"     # Renter-occupied
TOTAL_WORKERS_16P = "ASORE001"  # Total workers 16+ (denominator)
DRIVE_ALONE = "ASORE003"     # Drove alone
CARPOOLED = "ASORE004"     # Carpooled
PUBLIC_TRANSPORT = "ASORE010"     # Public transportation (excluding taxicab, all modes)
BICYCLE = "ASORE018"     # Bicycle
WALKED = "ASORE019"     # Walked
WORKED_FROM_HOME = "ASORE021"     # Worked from home
AUTO_COMMUTERS = "ASORE002"     # Car, truck, or van (all auto commuters)


tracts[[TRACT_GEOID_FIELD, TOTAL_POPULATION, MEDIAN_HOUSEHOLD_INCOME, MEDIAN_RENT, HOUSING_TOTAL, OCCUPIED_HOUSEHOLDS, VACANT_UNITS, OWNER_OCCUPIED, RENTER_OCCUPIED, TOTAL_WORKERS_16P, DRIVE_ALONE, CARPOOLED, PUBLIC_TRANSPORT, BICYCLE, WALKED, WORKED_FROM_HOME, AUTO_COMMUTERS]].head()

,GISJOIN,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E002,ASS9E003,ASORE001,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002
0,G0400130010102,6265,188486,-666666666,3537,2824,713,2731,93,2016,931,93,22,19,110,773,1024
1,G0400130010103,3681,117813,2744,1884,1516,368,1451,65,1735,1188,60,0,0,0,462,1248
2,G0400130010104,3131,140587,-666666666,2447,1737,710,1681,56,915,316,0,0,0,0,584,316
3,G0400130030401,4744,145865,1679,3170,2423,747,2221,202,1563,596,152,0,0,59,734,748
4,G0400130030402,4153,108031,2096,2273,1928,345,1803,125,1717,960,37,0,0,32,642,997


In [7]:
village.head()

,OBJECTID,NAME,geometry
0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598..."
1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551...."
2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860..."
3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89..."
4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89..."


In [8]:
tracts[[
    TRACT_GEOID_FIELD,
    TOTAL_POPULATION,
    MEDIAN_HOUSEHOLD_INCOME,
    MEDIAN_RENT
]].dtypes

GISJOIN     object
ASNQE001    object
ASQPE001    object
ASVBE001    object
dtype: object

In [9]:
# since they're objects, convert to numeric

acs_cols = [
    TOTAL_POPULATION, MEDIAN_HOUSEHOLD_INCOME, MEDIAN_RENT,
    HOUSING_TOTAL, OCCUPIED_HOUSEHOLDS, VACANT_UNITS,
    OWNER_OCCUPIED, RENTER_OCCUPIED,
    TOTAL_WORKERS_16P, DRIVE_ALONE, CARPOOLED, PUBLIC_TRANSPORT,
    BICYCLE, WALKED, WORKED_FROM_HOME, AUTO_COMMUTERS
]

for col in acs_cols:
    tracts[col] = pd.to_numeric(tracts[col], errors="coerce")

In [10]:
tracts.head()

,GISJOIN,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E001,ASS9E002,ASS9E003,ASORE001,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002,geometry
0,G0400130010102,6265,188486,-666666666,3537,2824,713,2824,2731,93,2016,931,93,22,19,110,773,1024,"POLYGON ((827801.708 1091626.91, 827865.552 10..."
1,G0400130010103,3681,117813,2744,1884,1516,368,1516,1451,65,1735,1188,60,0,0,0,462,1248,"POLYGON ((765755.318 1002785.693, 765592.367 1..."
2,G0400130010104,3131,140587,-666666666,2447,1737,710,1737,1681,56,915,316,0,0,0,0,584,316,"POLYGON ((765755.318 1002785.693, 765757.331 1..."
3,G0400130030401,4744,145865,1679,3170,2423,747,2423,2221,202,1563,596,152,0,0,59,734,748,"POLYGON ((707726.563 1037891.22, 707726.326 10..."
4,G0400130030402,4153,108031,2096,2273,1928,345,1928,1803,125,1717,960,37,0,0,32,642,997,"POLYGON ((707726.563 1037891.22, 707707.434 10..."


In [12]:
# time to begin spatial intersection with villages

import geopandas as gpd
import pandas as pd
import numpy as np

# Make a working copy of tracts with ACS
tracts_acs = tracts.copy()

tracts_acs = tracts_acs.to_crs(2868)
villages = village.to_crs(2868)

if "tract_acres" not in tracts_acs.columns:
    tracts_acs["tract_sqft"] = tracts_acs.geometry.area
    tracts_acs["tract_acres"] = tracts_acs["tract_sqft"] / 43560.0

In [13]:
villages.head()

,OBJECTID,NAME,geometry
0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598..."
1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551...."
2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860..."
3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89..."
4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89..."


In [14]:
tracts_acs.head()

,GISJOIN,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E001,ASS9E002,ASS9E003,...,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002,geometry,tract_sqft,tract_acres
0,G0400130010102,6265,188486,-666666666,3537,2824,713,2824,2731,93,...,931,93,22,19,110,773,1024,"POLYGON ((827801.708 1091626.91, 827865.552 10...",2.948847e+10,676962.212041
1,G0400130010103,3681,117813,2744,1884,1516,368,1516,1451,65,...,1188,60,0,0,0,462,1248,"POLYGON ((765755.318 1002785.693, 765592.367 1...",4.052815e+08,9303.984075
2,G0400130010104,3131,140587,-666666666,2447,1737,710,1737,1681,56,...,316,0,0,0,0,584,316,"POLYGON ((765755.318 1002785.693, 765757.331 1...",1.770686e+08,4064.934859
3,G0400130030401,4744,145865,1679,3170,2423,747,2423,2221,202,...,596,152,0,0,59,734,748,"POLYGON ((707726.563 1037891.22, 707726.326 10...",2.857097e+08,6558.993303
4,G0400130030402,4153,108031,2096,2273,1928,345,1928,1803,125,...,960,37,0,0,32,642,997,"POLYGON ((707726.563 1037891.22, 707707.434 10...",6.274937e+08,14405.273363


In [15]:
# Kill invalid medians (negative or crazy huge sentinel codes)
for col in [MEDIAN_HOUSEHOLD_INCOME, MEDIAN_RENT]:
    tracts_acs.loc[
        (tracts_acs[col] <= 0) | (tracts_acs[col] > 1_000_000),
        col
    ] = np.nan

In [16]:
tracts_x_village = gpd.overlay(
    tracts_acs,
    villages[["NAME", "geometry"]],
    how="intersection",
    keep_geom_type=False
)

# Intersection area in acres
tracts_x_village["intersect_sqft"] = tracts_x_village.geometry.area
tracts_x_village["intersect_acres"] = tracts_x_village["intersect_sqft"] / 43560.0

# Area weight - share of each tract's area that lies in this village
tracts_x_village["area_weight"] = (
    tracts_x_village["intersect_sqft"] /
    tracts_x_village["tract_sqft"]
)

# Sanity check: clamp weird round-off > 1
tracts_x_village.loc[tracts_x_village["area_weight"] > 1.0001, "area_weight"] = 1.0
tracts_x_village.loc[tracts_x_village["area_weight"] < 0, "area_weight"] = 0.0

In [17]:
# Count-like ACS fields to area-weight
COUNT_COLS = [
    TOTAL_POPULATION,
    HOUSING_TOTAL,
    OCCUPIED_HOUSEHOLDS,
    VACANT_UNITS,
    OWNER_OCCUPIED,
    RENTER_OCCUPIED,
    TOTAL_WORKERS_16P,
    DRIVE_ALONE,
    CARPOOLED,
    PUBLIC_TRANSPORT,
    BICYCLE,
    WALKED,
    WORKED_FROM_HOME,
    AUTO_COMMUTERS,
]

# Area-weighted counts
for col in COUNT_COLS:
    tracts_x_village[col + "_w"] = (
        tracts_x_village[col].fillna(0) *
        tracts_x_village["area_weight"].fillna(0)
    )

# Household-weighted medians (with area weighting)
# income_weighted ≈ Σ(median_income_tract * hh_tract * area_weight)
tracts_x_village["hh_weighted"] = (
    tracts_x_village[OCCUPIED_HOUSEHOLDS].fillna(0) *
    tracts_x_village["area_weight"].fillna(0)
)

tracts_x_village["income_weighted"] = (
    tracts_x_village[MEDIAN_HOUSEHOLD_INCOME].fillna(0) *
    tracts_x_village[OCCUPIED_HOUSEHOLDS].fillna(0) *
    tracts_x_village["area_weight"].fillna(0)
)

tracts_x_village["rent_weighted"] = (
    tracts_x_village[MEDIAN_RENT].fillna(0) *
    tracts_x_village[OCCUPIED_HOUSEHOLDS].fillna(0) *
    tracts_x_village["area_weight"].fillna(0)
)

In [18]:
agg_dict = {
    "intersect_acres": "sum",
    "hh_weighted": "sum",
    "income_weighted": "sum",
    "rent_weighted": "sum",
}
# Add all the count weights
agg_dict.update({col + "_w": "sum" for col in COUNT_COLS})

village_dem = (
    tracts_x_village
    .groupby("NAME", as_index=False)
    .agg(agg_dict)
)

In [19]:
# Approx median income & rent
village_dem["median_income_approx"] = np.where(
    village_dem["hh_weighted"] > 0,
    village_dem["income_weighted"] / village_dem["hh_weighted"],
    np.nan,
)
village_dem["median_rent_approx"] = np.where(
    village_dem["hh_weighted"] > 0,
    village_dem["rent_weighted"] / village_dem["hh_weighted"],
    np.nan,
)

# Unpack the counts (drop the _w suffix)
for col in COUNT_COLS:
    village_dem[col] = village_dem[col + "_w"]

# Per-acre densities
village_dem["pop_per_acre"] = village_dem[TOTAL_POPULATION] / village_dem["intersect_acres"]
village_dem["hh_per_acre"] = village_dem[OCCUPIED_HOUSEHOLDS] / village_dem["intersect_acres"]

# Mode shares (as percents)
den = village_dem[TOTAL_WORKERS_16P].replace(0, np.nan)

village_dem["pct_drive_alone"] = (village_dem[DRIVE_ALONE] / den) * 100
village_dem["pct_transit"]     = (village_dem[PUBLIC_TRANSPORT] / den) * 100
village_dem["pct_bike"]        = (village_dem[BICYCLE] / den) * 100
village_dem["pct_walk"]        = (village_dem[WALKED] / den) * 100
village_dem["pct_wfh"]         = (village_dem[WORKED_FROM_HOME] / den) * 100
village_dem["pct_auto"]        = (village_dem[AUTO_COMMUTERS] / den) * 100


In [20]:
village_dem.head(15)

,NAME,intersect_acres,hh_weighted,income_weighted,rent_weighted,ASNQE001_w,ASS8E001_w,ASS8E002_w,ASS8E003_w,ASS9E002_w,...,ASORE021,ASORE002,pop_per_acre,hh_per_acre,pct_drive_alone,pct_transit,pct_bike,pct_walk,pct_wfh,pct_auto
0,Ahwatukee Foothills,22832.880975,33008.638620,3.842066e+09,6.696994e+07,80121.395673,34930.417904,33008.638620,1921.779283,23107.487000,...,10231.748735,31444.695755,3.509036,1.445662,65.998273,0.497064,0.378675,1.030677,23.784279,73.094973
1,Alhambra,12315.416799,48938.870286,3.414169e+09,6.566797e+07,140247.377731,52861.579884,48938.870286,3922.709598,22623.919347,...,8008.939323,51103.297032,11.387952,3.973789,64.738882,4.149291,0.389546,2.628470,12.200772,77.850467
2,Camelback East,23242.629344,67179.084467,5.651596e+09,9.750743e+07,145914.152792,74575.675981,67179.084467,7396.591514,31318.147631,...,16077.929883,63175.490127,6.277868,2.890339,66.629334,2.594545,0.699733,1.763933,18.957796,74.491435
3,Central City,13600.592409,24096.557696,1.260087e+09,3.052167e+07,58532.832473,27345.355650,24096.557696,3248.797954,5997.054184,...,4866.687963,20468.800275,4.303697,1.771729,59.867798,4.629652,1.118289,4.122824,16.984972,71.437084
4,Deer Valley,36235.247205,73905.358172,6.444375e+09,1.262797e+08,185957.738644,77206.428216,73905.358172,3301.070044,47060.165824,...,18610.868578,74706.553066,5.131957,2.039599,66.321359,0.916779,0.264754,1.786102,19.035458,76.410911
5,Desert View,43415.141765,27724.807056,3.705466e+09,6.384562e+07,68812.687779,29776.482389,27724.807056,2051.675334,20146.126048,...,10459.362589,24295.172802,1.584993,0.638598,62.792894,0.137238,0.094896,0.818113,29.209149,67.847474
6,Encanto,6706.535758,26852.862637,1.927906e+09,3.828134e+07,57431.914093,30149.322941,26852.862637,3296.460303,10813.477157,...,6162.638237,23116.003408,8.563574,4.003984,61.704572,3.963402,0.983515,2.748542,19.001248,71.273519
7,Estrella,26503.624912,26650.951390,2.003729e+09,4.212900e+07,103023.425643,27735.695552,26650.951390,1084.744162,16828.846503,...,4308.713629,38520.883653,3.887145,1.005559,68.506736,1.983722,0.161181,0.523910,9.572126,85.576993
8,Laveen,19579.686598,19775.759721,2.020043e+09,3.508323e+07,68987.873610,20401.024379,19775.759721,625.264658,15500.707650,...,5712.796992,25274.902519,3.523441,1.010014,67.500289,1.577105,0.221852,0.641848,17.759230,78.571462
9,Maryvale,20820.532995,66558.962523,4.410263e+09,9.477484e+07,230776.337539,69275.719190,66558.962523,2716.756667,36717.338517,...,7631.946947,89851.926820,11.084074,3.196794,68.751150,2.162034,0.661646,0.995963,7.309967,86.061215


In [21]:
village_dem[["NAME", "median_income_approx", "median_rent_approx"]]

,NAME,median_income_approx,median_rent_approx
0,Ahwatukee Foothills,116395.759419,2028.861097
1,Alhambra,69763.953016,1341.836668
2,Camelback East,84127.309741,1451.455121
3,Central City,52293.256403,1266.640455
4,Deer Valley,87197.664217,1708.667261
5,Desert View,133651.640498,2302.833823
6,Encanto,71795.171522,1425.596374
7,Estrella,75184.153423,1580.769165
8,Laveen,102147.436370,1774.052390
9,Maryvale,66261.001213,1423.923089


In [23]:
# after generations of trying to figure out why approximated median rents and incomes were so negative... we have finally cracked it. time to export out.

village_dem_path = BOUNDARIES_DIR / "villages" / "villages_demographics_cleaned.csv"

village_dem.to_csv(village_dem_path, index=False)
print("Saved village demographics:", village_dem_path)

Saved village demographics: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\villages\villages_demographics_cleaned.csv
